# NUS SMC Corpus — Data Study

Exploratory analysis of the speech (`read/`) vs singing (`sing/`) corpus:
durations, phone inventory, speech↔singing duration ratios, and F0 ranges.

Run `scripts/preprocess.py` first to populate the feature cache and split manifest.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

with open(ROOT / "config" / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)

from data.nus_smc import NUS_SMC_Dataset
from data.splits import list_speakers, speaker_disjoint_split

sns.set_theme(style="whitegrid")
%matplotlib inline

ModuleNotFoundError: No module named 'numpy'

In [ ]:
split_map = speaker_disjoint_split(
    list_speakers(data_cfg["data_root"]),
    seed=data_cfg["splits"]["seed"],
    n_train=data_cfg["splits"]["train_speakers"],
    n_val=data_cfg["splits"]["val_speakers"],
)
ds = NUS_SMC_Dataset(data_cfg["data_root"], split_map, utterance_ids=None)
df = ds.to_dataframe()
print(f"items: {len(ds)}")
df.groupby("split")["speaker_id"].nunique().to_frame("n_speakers").join(
    df.groupby("split")["utt_id"].count().to_frame("n_utterances")
)

## 1. Phone-level duration table

Flatten every phone instance from both `read/` and `sing/` annotations.

In [ ]:
rows = []
for it in ds:
    for p in it["read_phones"]:
        rows.append({"speaker": it["speaker_id"], "utt": it["utt_id"],
                     "mode": "speech", "phone": p["phone"], "dur": p["duration"]})
    for p in it["sing_phones"]:
        rows.append({"speaker": it["speaker_id"], "utt": it["utt_id"],
                     "mode": "singing", "phone": p["phone"], "dur": p["duration"]})
ptab = pd.DataFrame(rows)
print("total phone instances:", len(ptab))
ptab.head()

## 2. Phone duration distribution: speech vs singing

Singing vowels are stretched across notes, so singing phone durations
should have a longer tail than speech.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for mode, color in [("speech", "tab:blue"), ("singing", "tab:orange")]:
    d = ptab[ptab["mode"] == mode]["dur"]
    d = d[d < 1.0]  # trim silence outliers for the plot
    ax.hist(d, bins=80, alpha=0.55, density=True, label=f"{mode} (median {d.median():.3f}s)", color=color)
ax.set_xlabel("phone duration (s)"); ax.set_ylabel("density")
ax.set_title("Phone duration distribution (trimmed < 1s)"); ax.legend()
plt.show()

## 3. Singing / speech duration ratio per phone

Which phones stretch the most when going from speech to singing (expect
vowels > consonants).

In [ ]:
piv = ptab.pivot_table(index=["speaker", "utt", "phone"], columns="mode",
                       values="dur", aggfunc="mean")
ratio = piv.assign(ratio=piv["singing"] / piv["speech"].clip(lower=1e-6)).dropna(subset=["ratio"])
med = ratio.groupby("phone")["ratio"].median().sort_values(ascending=False)

top = med.head(15)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(top)), top.values, color="tab:orange")
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index)
ax.invert_yaxis(); ax.set_xlabel("median singing/speech duration ratio")
ax.set_title("Top 15 phones by duration stretch (speech → singing)")
plt.show()

## 4. Phone inventory

ARPAbet phone types present in the corpus.

In [ ]:
counts = ptab["phone"].value_counts()
print("unique phone types:", len(counts))
fig, ax = plt.subplots(figsize=(11, 5))
counts.head(25).plot(kind="bar", ax=ax, color="tab:green")
ax.set_ylabel("count"); ax.set_title("Top 25 phone types")
plt.show()

## 5. Speaker balance

Number of utterances per speaker and their assigned split.

In [ ]:
bal = df.groupby("speaker_id").agg(n_utts=("utt_id", "count"), split=("split", "first"))
bal

## 6. Speech vs singing F0 (from cached CREPE for singing + on-the-fly for speech)

Singing has a wider and typically higher F0 range than read speech.

In [ ]:
from features.cache import FeatureCache
from features.crepe_f0 import extract_f0

cache = FeatureCache(data_cfg["cache_root"])
f0_rows = []
for it in ds[:8]:
    key = f"{it['speaker_id']}_{it['utt_id']}"
    try:
        f0_sing = cache.load("crepe", it["split"], key)
        conf_sing = cache.load("crepe_conf", it["split"], key)
        f0_read, conf_read = extract_f0(it["read_wav"], model_capacity="tiny", hop_length=160, device="auto")
        for label, f0, conf in [("speech", f0_read, conf_read), ("singing", f0_sing, conf_sing)]:
            v = f0[conf >= 0.5]  # torchcrepe never outputs 0 Hz
            if len(v):
                f0_rows.append({"mode": label, "median_hz": float(np.median(v)),
                                "min_hz": float(v.min()), "max_hz": float(v.max())})
    except Exception as e:
        print("skip", key, e)
f0df = pd.DataFrame(f0_rows)

fig, ax = plt.subplots(figsize=(9, 4))
for mode, color in [("speech", "tab:blue"), ("singing", "tab:orange")]:
    v = f0df[f0df["mode"] == mode]["median_hz"]
    ax.hist(v, bins=20, alpha=0.55, density=True, label=mode, color=color)
ax.set_xlabel("median voiced F0 (Hz)"); ax.set_ylabel("density")
ax.set_title("Median F0 per utterance (speech vs singing)"); ax.legend()
plt.show()

f0df.groupby("mode").agg(mean_hz=("median_hz", "mean"), min_hz=("min_hz", "min"), max_hz=("max_hz", "max"))

## 7. Sample annotation timeline

Overlay the phone boundaries of one `sing/` utterance to visually verify
the annotations.

In [ ]:
it = ds[0]
phones = it["sing_phones"]
fig, ax = plt.subplots(figsize=(12, 3))
for p in phones:
    ax.plot([p["start"], p["end"]], [0, 0], lw=3, solid_capstyle="butt")
    ax.text((p["start"] + p["end"]) / 2, 0.02, p["phone"], ha="center", fontsize=7, rotation=90)
ax.set_yticks([]); ax.set_xlabel("time (s)")
ax.set_title(f"singing phone timeline: {it['speaker_id']} / {it['utt_id']}")
plt.show()